## 1. RAG에서 텍스트 전처리의 중요성

RAG 파이프라인의 첫 번째 단계는 **Indexing(인덱싱)** 이었습니다. 이 단계에서 문서를 검색 가능한 형태로 변환하는 과정은 다음과 같습니다:

```
원본 문서 → 로딩 → 클리닝(전처리) → 청킹(분할) → 임베딩(벡터화) → 벡터 DB 저장
```

이 중 **청킹(Chunking)** 은 RAG 성능에 가장 큰 영향을 미치는 단계 중 하나입니다.

### 왜 청킹이 중요한가?

| 청크가 너무 작을 때 | 청크가 너무 클 때 |
|:--:|:--:|
| 문맥 정보가 손실됨 | 관련 없는 내용이 포함됨 |
| 의미가 불완전한 조각 | 검색 정확도 하락 |
| 검색 결과는 많지만 품질 낮음 | LLM 토큰 한도 초과 위험 |

따라서 **적절한 청크 크기와 분할 전략을 선택** 하는 것이 RAG 시스템의 핵심 설계 결정입니다.

In [ ]:
import re

# RAG 관련 한국어 기술 문서
sample_document = """
검색 증강 생성(RAG) 기술 개요

1. RAG의 정의와 배경

RAG(Retrieval-Augmented Generation)는 2020년 Meta AI 연구팀이 발표한 혁신적인 기술이다. 이 기술은 대규모 언어 모델(LLM)의 텍스트 생성 능력과 정보 검색 시스템을 결합한 하이브리드 접근법이다. 기존의 LLM은 학습 데이터에만 의존하여 답변을 생성했기 때문에, 학습 이후의 정보나 특정 도메인의 전문 지식에 대해서는 정확한 답변을 제공하기 어려웠다. RAG는 이러한 한계를 극복하기 위해, 질문이 주어지면 먼저 외부 지식 소스에서 관련 문서를 검색하고, 검색된 문서를 컨텍스트로 활용하여 답변을 생성한다.

2. 벡터 데이터베이스의 역할

벡터 데이터베이스는 RAG 시스템의 핵심 인프라이다. 텍스트 데이터를 고차원 벡터(임베딩)로 변환하여 저장하고, 코사인 유사도를 기반으로 의미적으로 유사한 문서를 빠르게 검색할 수 있게 해준다. 대표적인 벡터 데이터베이스로는 ChromaDB, FAISS, Pinecone, Weaviate, Milvus 등이 있다. 각 데이터베이스는 인덱싱 방식, 검색 속도, 확장성 면에서 고유한 특성을 가진다.

3. 임베딩 모델과 의미적 검색

임베딩(Embedding)은 텍스트를 고차원 수치 벡터로 변환하는 과정이다. 의미적으로 유사한 텍스트는 벡터 공간에서 가까이 위치하게 된다. OpenAI의 text-embedding-3-small 모델은 1536차원의 벡터를 생성하며, 다국어를 지원한다. 임베딩을 통해 키워드 매칭이 아닌 의미적 유사성 기반의 검색이 가능해진다.

4. 청킹 전략의 중요성

청킹(Chunking)은 긴 문서를 적절한 크기의 작은 단위로 분할하는 과정이다. 청크의 크기가 너무 작으면 문맥 정보가 손실되고, 너무 크면 검색 정확도가 떨어진다. 주요 청킹 전략으로는 고정 크기 청킹, 문장 기반 청킹, 재귀적 청킹 등이 있다. 각 전략은 문서의 특성과 사용 목적에 따라 적절히 선택해야 한다.

5. 고급 RAG 패턴

기본 RAG의 한계를 극복하기 위한 다양한 고급 패턴이 연구되고 있다. Query Rewriting은 사용자의 질문을 검색에 최적화된 형태로 변환하는 기법이다. HyDE(Hypothetical Document Embedding)는 질문에 대한 가상 답변을 생성하여 검색에 활용한다. Re-ranking은 초기 검색 결과를 재정렬하여 관련성을 높인다. Multi-Query는 하나의 질문에서 여러 검색 질의를 생성하여 검색 범위를 확장한다.

6. 지식 그래프와 Hybrid RAG

지식 그래프(Knowledge Graph)는 엔티티와 관계를 그래프 구조로 표현하는 방식이다. 벡터 검색이 의미적 유사성에 강한 반면, 지식 그래프는 구조적 관계 추론에 강하다. Hybrid RAG는 벡터 검색과 그래프 검색을 결합하여 두 가지 장점을 모두 활용하는 접근법이다. 이를 통해 의미적 유사성과 구조적 관계를 동시에 고려한 검색이 가능해진다.
"""

# 텍스트 클리닝 함수
def clean_text(text):
    """텍스트 전처리: 불필요한 공백과 특수문자 정리"""
    text = text.strip()  # 앞뒤 공백 제거
    text = re.sub(r'\n{3,}', '\n\n', text)  # 연속 줄바꿈 정리
    text = re.sub(r' {2,}', ' ', text)  # 연속 공백 정리
    return text

cleaned_text = clean_text(sample_document)

print(f"원본 문서 길이: {len(sample_document)}자")
print(f"클리닝 후 길이: {len(cleaned_text)}자")
print(f"줄 수: {len(cleaned_text.splitlines())}줄")
print()
print("[문서 미리보기 (처음 200자)]")
print(cleaned_text[:200] + "...")